In [ ]:
import pandas as pd
from tqdm import tqdm
import numpy as np
from lifelines import CoxPHFitter
import matplotlib.pyplot as plt

In [ ]:
import Data_import

In [ ]:
PCDM_DEMOGRAPHIC = Data_import.PCDM_DEMOGRAPHIC
PCDM_DEATH_CAUSE = Data_import.PCDM_DEATH_CAUSE
PCDM_DEATH = Data_import.PCDM_DEATH
PCDM_DIAGNOSIS = Data_import.PCDM_DIAGNOSIS
PCDM_ENCOUNTER = Data_import.PCDM_ENCOUNTER
PCDM_IMMUNIZATION = Data_import.PCDM_IMMUNIZATION
PCDM_LAB_RESULT_CM = Data_import.PCDM_LAB_RESULT_CM
PCDM_LDS_ADDRESS_HISTORY = Data_import.PCDM_LDS_ADDRESS_HISTORY
PCDM_MED_ADMIN = Data_import.PCDM_MED_ADMIN
PCDM_OBS_CLIN = Data_import.PCDM_OBS_CLIN
PCDM_OBS_GEN = Data_import.PCDM_OBS_GEN
PCDM_PRESCRIBING = Data_import.PCDM_PRESCRIBING
PCDM_PROCEDURES = Data_import.PCDM_PROCEDURES
PCDM_PROVIDER = Data_import.PCDM_PROVIDER
PCDM_VITAL = Data_import.PCDM_VITAL

In [ ]:
### Translation dictionary for diagnosis codes
DiagCodeTranslationDF = pd.read_csv('ICD_RX translation/HNC Diagnosis ICD Translation DF.csv')
diagTranslateDict=dict(zip(DiagCodeTranslationDF['ICD_CODES'], DiagCodeTranslationDF['DESCRIPTION']))

### Translation dictionary for procedures
ProcedCodeTranslationDf = pd.read_csv('ICD_RX translation/HNC procedure ICD Translation DF.csv')
procedTranlateDict=dict(zip(ProcedCodeTranslationDf['ICD_CODES'], ProcedCodeTranslationDf['DESCRIPTION']))

### Translation dictionary for rxnorm cui codes
column_types = {'RX_CODE':str, 'TRANSLATION':str}
rxCodeTranslatedf = pd.read_csv('ICD_RX translation/rxCodeTranslationDF_HNC.csv', dtype=column_types)
rxCodeTranslatedf['TRANSLATION'] = rxCodeTranslatedf['TRANSLATION'].fillna('')
rxCodeTranslateDict =dict(zip(rxCodeTranslatedf['RX_CODE'], rxCodeTranslatedf['TRANSLATION']))

### Translation dictionary for NDC med codes
column_types = {'NDC_CODE':str, 'TRANSLATION':str}
ndcTranslatedf = pd.read_csv('ICD_RX translation/ndcCodeTranslationDF_HNC.csv', dtype=column_types)
ndcTranslatedf['TRANSLATION'] = ndcTranslatedf['TRANSLATION'].fillna('')
ndcCodeTranslateDict =dict(zip(ndcTranslatedf['NDC_CODE'], ndcTranslatedf['TRANSLATION']))

### PCDM_DIAGNOSIS/ PCDM_DEMOGRAPHIC

#### Patient data

To extract key patient information:

- age at HNC diagnosis
- date of HNC diagnosis
- gender
- metastasis status/ date of metastasis diagnosis

to extract diagnosis codes:

- diagnosis codes
- date of diagnosis

##### AGE at diagnosis/ HNC date of diagnosis

In [ ]:
list(PCDM_DIAGNOSIS['PATID'].unique())

In [ ]:
PCDM_DEMOGRAPHIC.head()

In [ ]:
pat_gender = {}
for pat in tqdm(PCDM_DEMOGRAPHIC['PATID'].unique()):
    if PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['SEX'].values[0] == 'M':
        pat_gender[pat] = 1
    else:
        pat_gender[pat] = 0


In [ ]:
pd.DataFrame(columns = ['PCDM_PATID', 'AGE_AT_DIAGNOSIS', 'DATE_OF_HNC_DIAGNOSIS', 'GENDER', 'METASTASIS_STATUS', 'DATE_OF_METASTASIS'])

### Get age at diagnosis
### identify date of HNC diagnosis
HNC_ICD_9 = [140, 141, 142, 143, 144, 145, 146, 147, 148, 149]
HCN_ICD_10 = ['C00', 'C01', 'C02', 'C03', 'C04', 'C05', 'C06', 'C07', 'C08', 'C09', 'C10', 'C11', 'C12', 'C13', 'C14', 'C31', 'C80.1','C76', 'C44.42', 'C30', 'C31', 'C32', 'C33']

ICD_9_diag = []
for val in PCDM_DIAGNOSIS['DX']:
    for val2 in HNC_ICD_9:
        if str(val2) in str(val):
            #print(val)
            ICD_9_diag.append(val)

ICD_10_diag = []
for val in PCDM_DIAGNOSIS['DX']:
    for val2 in HCN_ICD_10:
        if str(val2) in str(val):
            #print(val)
            ICD_10_diag.append(val)

HNC_DIAG = PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['DX'].isin(ICD_9_diag + ICD_10_diag)]
HNC_DIAG_PAT = {}
for pat in tqdm(PCDM_DIAGNOSIS['PATID'].unique()):
    # print(pat)
    # print(HNC_DIAG[HNC_DIAG['PATID']==pat]['DX'])
    # print(HNC_DIAG[HNC_DIAG['PATID']==pat]['DX_DATE'])
    # print(HNC_DIAG[HNC_DIAG['PATID']==pat]['DX_DATE'].min())
    HNC_DIAG_PAT[pat] = HNC_DIAG[HNC_DIAG['PATID']==pat]['DX_DATE'].min()

hnc_diagnosis_date_data = pd.DataFrame.from_dict(HNC_DIAG_PAT, orient = 'index',columns=['HNC_DIAG_DATE']).reset_index(names='PATID')
drop_pat = list(set(hnc_diagnosis_date_data[hnc_diagnosis_date_data['HNC_DIAG_DATE'].isnull()]['PATID']))

age_at_diagnosis = {}
for pat in tqdm(PCDM_DIAGNOSIS['PATID'].unique()):
    # print(pat)
    # print(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PCDM_PATID']==pat]['BIRTH_DATE'])
    # print(HNC_DIAG_PAT[pat])
    # print(pat)
    # print('diag')
    # print(pd.to_datetime(HNC_DIAG_PAT[pat]))
    # print('demo')
    # print(pd.to_datetime(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['BIRTH_DATE']))
          
    # print((pd.to_datetime(HNC_DIAG_PAT[pat])- pd.to_datetime(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['BIRTH_DATE'])).dt.days/365.25)
    try:
        age_at_diagnosis[pat] =  int( (pd.to_datetime(HNC_DIAG_PAT[pat])- pd.to_datetime(PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['BIRTH_DATE'])).dt.days/365.25 )
    except:
        age_at_diagnosis[pat] = np.nan 

age_data=pd.DataFrame.from_dict(age_at_diagnosis, orient = 'index',columns=['AGE_AT_DIAGNOSIS']).reset_index(names='PATID')

pat_gender = {}
for pat in tqdm(PCDM_DEMOGRAPHIC['PATID'].unique()):
    if PCDM_DEMOGRAPHIC[PCDM_DEMOGRAPHIC['PATID']==pat]['SEX'].values[0] == 'M':
        pat_gender[pat] = 1
    else:
        pat_gender[pat] = 0

pat_gender_data = pd.DataFrame.from_dict(pat_gender, orient = 'index',columns=['GENDER']).reset_index(names='PATID')


##### Metastasis status/ outocme date

In [ ]:
metastasisICD = []
for val in tqdm(diagTranslateDict.keys()):
    #print(str.lower(str(translateDict[val])))
    if 'metas' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'spread' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary unspecified malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)
    if 'secondary and unspecified malignant neoplasm' in str.lower(str(diagTranslateDict[val])):
        metastasisICD.append(val)

### Get metastasis status
onlyMetData = PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['DX'].isin(metastasisICD)]
### Create list of patients with metastasis ICD codes
metPatients=list(set(onlyMetData['PATID']))
### Get date of metastasis
patid = []
outcome = []
outDate = []
for pat in tqdm(PCDM_DIAGNOSIS['PATID'].unique()):
    patid.append(pat)
    if pat in metPatients:
        outDate.append( onlyMetData[onlyMetData['PATID']==pat]['DX_DATE'].min() )
        outcome.append(1)
    else:
        outDate.append('MAX')
        outcome.append(0)

outcomeData = {
    'PATID': patid,
    'METASTASIS': outcome,
    'OUTCOME_DATE': outDate
}       
metastasis_data = pd.DataFrame(outcomeData)

print(len(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID'])))
print(len(set(metastasis_data[metastasis_data['METASTASIS']==0]['PATID'])))
print(len(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID']))/ len(set(metastasis_data['PATID'])))

##### metastasis graphing

In [ ]:
df = onlyMetData
# Determine the unique number of patients diagnosed with each diagnosis code
unique_patients_by_code = df.groupby('DX')['PATID'].nunique().reset_index()
unique_patients_by_code.columns = ['DX', 'NumberOfUniquePatients']
# Plotting a bar graph of unique patients by diagnosis code
top_met = unique_patients_by_code.sort_values(by='NumberOfUniquePatients', ascending=False).head(20)
top_met = top_met.set_index('DX')

# top_met.plot(  kind='bar', color='#1c77b4', figsize=(8, 6))
# Determine the unique number of patients diagnosed with each diagnosis code

plt.figure(figsize=(8, 6))
plt.bar(top_met.index, top_met['NumberOfUniquePatients'], color='#1c77b4')
plt.title('Unique Patients by Diagnosis Code')
plt.xlabel('Diagnosis Code')
plt.ylabel('Number of Unique Patients')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
unique_patients_by_code
translations = []
for val in unique_patients_by_code['DX']:
    translations.append(diagTranslateDict[val])
unique_patients_by_code['TRANSLATION'] = translations

In [ ]:
pd.set_option('display.max_colwidth', None)
unique_patients_by_code.sort_values(by = 'NumberOfUniquePatients', ascending=False).head(20)

#### diagnosis comorbidity data

In [ ]:
### outcome date correct pcdm_med_admin
print(len(PCDM_DIAGNOSIS))
PCDM_DIAGNOSIS['DX_DATE'] = pd.to_datetime(PCDM_DIAGNOSIS['DX_DATE'], errors = 'coerce')
PCDM_DIAGNOSIS = PCDM_DIAGNOSIS.dropna(subset = ['DX_DATE'])
print(len(PCDM_DIAGNOSIS))
metPatients = list(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID']))
data_rows = []
parse_pat = [p for p in PCDM_DIAGNOSIS['PATID'].unique() if p not in drop_pat]
for pat in tqdm(parse_pat):
    temp = PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['PATID']==pat]
    # Get HNC diagnosis date
    hnc_diag_date = HNC_DIAG_PAT.get(pat)
    if pd.notna(hnc_diag_date):
        temp = temp[temp['DX_DATE'] >= pd.to_datetime(hnc_diag_date)]
    if pat in metPatients:
        outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
        append_rows = temp[temp['DX_DATE']<= outcome_date]
        data_rows.append(append_rows)
    else:
        data_rows.append(temp)
    
outcome_corrected_PCDM_DIAGNOSIS = pd.concat(data_rows, ignore_index=True)

In [ ]:
HNC_DIAG_CODES = ICD_9_diag + ICD_10_diag
metastasis_codes = metastasisICD
comorbididty_data = outcome_corrected_PCDM_DIAGNOSIS[~outcome_corrected_PCDM_DIAGNOSIS['DX'].isin(HNC_DIAG_CODES + metastasis_codes)]
#comorbididty_data['DX_DATE'] = pd.to_datetime(comorbididty_data['DX_DATE'])
comorbididty_data = comorbididty_data[['PATID', 'DX', 'DX_DATE']]
comorbididty_data

### PCDM_PROCEDURE

to extract key patient information:
- Was surgery conducted
- Was radiation treatment conducted

to extract procedure codes

- procedure codes
- date of procedure


**Need to ensure all information included is before the metastasis outcome**

In [ ]:
PCDM_PROCEDURES.head()

#### Limit to before outcome date and after diagnosis date

In [ ]:
### isolate to only below otucome date
# surgery_data = PCDM_PROCEDURES[PCDM_PROCEDURES['PX'].isin(top_surgery_codes)]
PCDM_PROCEDURES['PX_DATE'] = pd.to_datetime(PCDM_PROCEDURES['PX_DATE'], errors = 'coerce')
PCDM_PROCEDURES = PCDM_PROCEDURES.dropna(subset = ['PX_DATE'])
metastasis_data['OUTCOME_DATE'] = pd.to_datetime(metastasis_data['OUTCOME_DATE'], errors = 'coerce')
data_rows = []
parse_pat = [p for p in PCDM_PROCEDURES['PATID'].unique() if p not in drop_pat]
for pat in tqdm(parse_pat):
    temp = PCDM_PROCEDURES[PCDM_PROCEDURES['PATID']==pat]
    ## get HNC diagnosis date
    hnc_diag_date = HNC_DIAG_PAT.get(pat)
    if pd.notna(hnc_diag_date):
        temp = temp[temp['PX_DATE'] >= pd.to_datetime(hnc_diag_date)]
    
    ### get metastasis outcome
    if pat in metPatients:
        outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
        append_rows = temp[temp['PX_DATE']<=outcome_date]
        data_rows.append(append_rows)
    else:
        data_rows.append(temp)
outcome_corrected_PCDM_PROCEDURES = pd.concat(data_rows, ignore_index=True)

#### identify surgery/ radiation status

In [ ]:
### Calculated in ../Procedures/Identify srugery prevalance
# These procedure codes were included to broadly capture surgical and procedure-related interventions
# that may be associated with medication use and thus act as potential confounders in downstream analyses.
# The list intentionally spans multiple categories—including definitive surgical procedures (e.g., resections,
# neck dissections, airway surgeries), perioperative or supportive interventions (e.g., tracheostomy, feeding
# tube placement), pathology/laboratory processing (e.g., 883xx series), and ancillary HCPCS/supply or drug
# administration codes—to ensure comprehensive coverage of the clinical workflow surrounding treatment.
# While not all codes directly represent therapeutic interventions, each may reflect a clinical state,
# complication risk, or care pathway (e.g., infection risk, pain burden, nutritional support, airway management,
# or postoperative monitoring) that influences medication utilization and patient outcomes. Therefore, inclusion
# of these heterogeneous procedure groups allows for more complete identification and adjustment of confounding
# factors, improving the interpretability and robustness of associations between treatments and outcomes.
top_surgery_codes = ['88305',
 '99024',
 '88307',
 '88331',
 '38724',
 '88309',
 '88311',
 '49440',
 '31600',
 '88304',
 '43653',
 '43246',
 '41120',
 '49450',
 '88332',
 '43762',
 '88300',
 '31237',
 'S2900',
 '31502',
 '17000',
 '41150',
 '43.11',
 '17311',
 '17003',
 '42826',
 '35701',
 '17110',
 '31.1',
 '31360',
 '42890',
 '31225',
 '41130',
 '17312',
 '49465',
 '97.02',
 '31615',
 '25.2',
 '41153',
 '69433',
 '88333',
 '00142',
 '61590',
 '38700',
 '47562',
 '43830',
 '61581',
 '88302',
 '32666',
 '21070',
 '76.31',
 '31267',
 '41155',
 '45.42',
 'L8501',
 '31040',
 '93286',
 '88334',
 '32674',
 '32551',
 '43030',
 '49650',
 '43.19',
 '17004',
 '69436',
 '21470',
 '17262',
 '49452',
 '29.33',
 '19301',
 '41135',
 '49446',
 '31276',
 '15839',
 '32663',
 '93287',
 '23.19',
 '31288',
 '28.2',
 '38720',
 '76098',
 '67414',
 '33508',
 '97.23',
 '43242',
 '15004',
 '31255',
 '17313',
 '61796',
 '60240',
 '21198',
 '60252',
 '41140',
 '13160',
 '60220',
 '29826',
 '42960',
 '17261',
 '31367',
 '21047',
 '43274',
 '61510',
 '66982',
 '30150',
 '31603',
 '21685',
 '26.31',
 '92941',
 '26.32',
 '69420',
 '52234',
 '29827',
 '22614',
 '17272',
 '69990',
 '22551',
 '92626',
 '20930',
 'G0416',
 '61583',
 '43262',
 '66821',
 '63047',
 '17263',
 '22.63',
 '43259',
 '43870',
 '35301',
 '63030',
 '49451',
 '42962',
 '31259',
 '51703',
 '22853',
 '31610',
 '77372',
 '44180',
 '31238',
 '22.64',
 'J9280',
 '37225',
 '19303',
 '30.4',
 '29823',
 '29881',
 'G0269',
 '17282',
 '32668',
 '99195',
 '44186',
 '15002',
 '30.3',
 '60500',
 '31395',
 '38745',
 '32667',
 '52235',
 '44160',
 '61591',
 '52601',
 '44205',
 '31365',
 '21046',
 '44213',
 '44970',
 '29824',
 '63048',
 '51595',
 '22552',
 '31254',
 '37799',
 'G8918',
 '31825',
 'C1714',
 '61580',
 '29.31',
 '55866',
 '58571',
 '41830',
 '22600',
 '52224',
 '44015',
 '68720',
 '20985',
 '35371',
 '29880',
 'L3260',
 '97.03',
 '61645',
 '15003',
 '31605',
 '20936',
 '52240',
 '29828',
 '17271',
 '61797',
 '01402',
 '50432',
 '00103',
 '49441',
 'C1724',
 '00.33',
 'A4550',
 '31256',
 '43116',
 '47600',
 '67412',
 '17314',
 '39.61',
 '67036',
 '67108',
 '67400',
 '55840',
 '51.23',
 '17273',
 '69502',
 '44139',
 '44120',
 '83.45',
 '38570',
 '32652',
 '32650',
 '31253',
 '31820',
 '00.31',
 '52281',
 '06.4',
 '58558',
 '58661',
 '22.2',
 '20931',
 '17281',
 '50205',
 '92933',
 '48.36',
 '44207',
 '25.4',
 'A4649',
 '01630',
 '44145',
 'C1757',
 'J7315',
 '43763',
 '00176',
 '26.30',
 '51.85',
 '51045',
 '52648',
 '69930',
 '22.41',
 '19307',
 '15005',
 '97.51',
 '50360',
 '30.29',
 '49651',
 '46221',
 '30580',
 '31.74',
 '42140',
 '44300',
 '44310',
 '44625',
 '45110',
 '23430',
 '46.39',
 '32480',
 '47370',
 '49321',
 '47563',
 '49000',
 '40500',
 '43281',
 '43644',
 '32651',
 '32653',
 'J0878',
 '44140',
 '42821',
 '32669',
 '44204',
 '38746',
 '92937',
 '61518',
 '56620',
 '86612',
 '38571',
 '69421',
 '58573',
 '46.32',
 '63620',
 '63045',
 '36832',
 '67042',
 '43276',
 '19340',
 '17315',
 '31390',
 '01400',
 '28285',
 '30160',
 '22612',
 '31830',
 '23120',
 '32.41',
 '22610',
 '17283',
 '00.39',
 '92.32',
 '88329',
 '50435',
 '50543',
 '57283',
 '43107',
 '22633',
 '42961',
 '92.30',
 '29876',
 '31287',
 '06.2',
 '92943',
 '63277',
 '42.41',
 '42330',
 '50389',
 '21085',
 '85.23',
 '66852',
 '27310',
 '27132',
 '27602',
 '17276',
 '69530',
 '46922',
 '46924',
 '28008',
 '44188',
 '47143',
 '69643',
 '17264',
 '44143',
 '48150',
 '44050',
 '17260',
 '49324',
 '49460',
 '58262',
 '63015',
 '45990',
 '60210',
 '60260',
 'G8916',
 '36831',
 '60650',
 '60.5',
 '61585',
 '00540',
 '20937',
 '59400',
 '61798',
 '37187',
 '37227',
 '61312',
 '38.82',
 '35372',
 '32.49',
 '21048',
 '21365',
 '22.52',
 '25.3',
 '69644',
 '43775',
 '77.81',
 '44150',
 '67335',
 '50693',
 '17111',
 '28750',
 '43832',
 '49002',
 '76.64',
 '44.38',
 '25020',
 '76.39',
 '17.35',
 '19302',
 '67041',
 '50546',
 '43253',
 '35875',
 '01.25',
 '17.56',
 '44141',
 '69635',
 'C9602',
 '19342',
 '61618',
 '34201',
 '60271',
 '28122',
 '67311',
 '21620',
 '22854',
 '31370',
 '44372',
 '54161',
 '17270',
 '31230',
 '39.27',
 '19371',
 '46255',
 '31240',
 '32120',
 '67825',
 '61582',
 '63035',
 '00126',
 '21280',
 '20.49',
 '38.18',
 '54065',
 '21050',
 '21049',
 '36833',
 '22585',
 '32110',
 '52276',
 '51705',
 '61500',
 '41820',
 '44320',
 '14.74',
 '67450',
 '54056',
 '44202',
 '10040',
 '47.01',
 '46.20',
 '47120',
 '26037',
 '17284',
 '17286',
 '46270',
 '67445',
 '46260',
 '61320',
 '21282',
 '58260',
 '67040',
 '20.01',
 '63267',
 '54.11',
 '63265',
 '54060',
 '62121',
 '63081',
 '55.51',
 '63046',
 '22554',
 '56.51',
 '56501',
 '56515',
 '22206',
 '56630',
 '57.17',
 '57.71',
 '58662',
 '62100',
 '19316',
 '65855',
 '49659',
 '50230',
 '61697',
 '50545',
 '50548',
 '50688',
 '50695',
 '65820',
 '19370',
 '51.22',
 '65400',
 '22630',
 '60.29',
 '64774',
 '69220',
 '22595',
 '52.7',
 '51710',
 '45397',
 '86.24',
 '38.83',
 '43.99',
 '29822',
 '43112',
 '43117',
 '31300',
 '28725',
 '85.6',
 '16.09',
 '37765',
 '37229',
 'A7526',
 '37184',
 '43332',
 '85.0',
 '43420',
 'A4369',
 '38120',
 '31375',
 '31070',
 '0275T',
 '06.39',
 '31239',
 '09.81',
 '31087',
 '31084',
 '41145',
 '31051',
 '15850',
 '31241',
 '42335',
 '15829',
 '97.62',
 '38564',
 '42825',
 '31.29',
 '36905',
 '31200',
 '27707',
 '32655',
 '17266',
 '33999',
 '73530',
 '44346',
 '32662',
 '32659',
 'C9607',
 '44227',
 '27870',
 '44121',
 '28124',
 '76.41',
 '17280',
 '44210',
 '00865',
 '45.73',
 'C9604',
 '77.88',
 '45.76',
 '45190',
 '31382',
 'G8911',
 '28288',
 '69636',
 '97.41',
 '00542',
 '01951',
 '21.69',
 '0238T',
 '97.04',
 '20.42',
 '00162',
 '00.35',
 '96.36',
 '63042',
 '63003',
 '0184T',
 '63020',
 '01638',
 '01740',
 '61619',
 'C9605',
 '93750',
 '01756',
 'A4625',
 'C9606',
 '62201',
 '01830',
 '62350',
 '61597',
 '61592',
 '63001',
 'C9766',
 '01274',
 '61698',
 '01842',
 '61586',
 '20.92',
 '63741',
 '63082',
 '67314',
 '17.55',
 '67010',
 '67015',
 '83.13',
 '83.02',
 '80.05',
 '19305',
 '77.89',
 '67110',
 '67113',
 '76.65',
 '67312',
 '67318',
 '63102',
 '67334',
 '19020',
 '69633',
 '69631',
 '67420',
 '69604',
 '21194',
 '69511',
 '67715',
 '17274',
 '68.41',
 '68550',
 '85.34',
 '66761',
 '66635',
 '85.41',
 '07.22',
 '92627',
 '63275',
 '63276',
 '63300',
 '15828',
 '63303',
 '92.31',
 '68760',
 '15830',
 '64738',
 '15832',
 '65.41',
 '15877',
 '15878',
 '65771',
 '16035',
 '16036',
 '17.32',
 '88249',
 '17.36',
 '66.62',
 '85.44',
 '66170',
 '85.43',
 '69601',
 '32505',
 '61530',
 '43130',
 '43282',
 '28289',
 '43280',
 '43277',
 '00.32',
 '28291',
 '28300',
 '28306',
 '43180',
 '29806',
 '43335',
 '43.3',
 '29825',
 '29848',
 '42970',
 '29874',
 '42830',
 '31030',
 '42820',
 '42340',
 '43287',
 '43352',
 '61514',
 '44020',
 '28.7',
 '44187',
 '28010',
 '28060',
 '28111',
 '44158',
 '28112',
 '28118',
 '44021',
 '28140',
 '43620',
 '44.32',
 '28150',
 '28230',
 '43820',
 '43774',
 '28232',
 '28270',
 '43633',
 '43632',
 '31050',
 '42.52',
 '42.42',
 '31420',
 '37.11',
 '36906',
 '35876',
 '35351',
 '35303',
 '34111',
 '34101',
 '34.03',
 '33572',
 '33366',
 '42.11',
 '33202',
 '33030',
 '32671',
 '32036',
 '32097',
 '32320',
 '32442',
 '32445',
 '32560',
 '37185',
 '37186',
 '31368',
 '37766',
 '31080',
 '31081',
 '31085',
 '31090',
 '41010',
 '41.5',
 '40819',
 '39010',
 '38770',
 '38760',
 '38747',
 '38740',
 '31257',
 '38572',
 '31292',
 '31295',
 '38562',
 '38100',
 '38.85',
 '27709',
 '27610',
 '44322',
 '54530',
 '57.18',
 '22208',
 '55873',
 '22212',
 '55845',
 '22216',
 '55250',
 '55.4',
 '54860',
 '54057',
 '51550',
 '53215',
 '52649',
 '22222',
 '52630',
 '22556',
 '22558',
 '22590',
 '52214',
 '32486',
 '57065',
 '58146',
 '58150',
 '58552',
 '61501',
 '61343',
 '61304',
 '60540',
 '60521',
 '60520',
 '60502',
 '60254',
 '21433',
 '60225',
 '21750',
 '60212',
 '59409',
 '59151',
 '59.94',
 '58720',
 '22.31',
 '22.42',
 '58563',
 '51590',
 '51040',
 '44373',
 '46060',
 '47490',
 '26110',
 '47125',
 '46946',
 '46945',
 '46916',
 '46280',
 '46258',
 '26850',
 '26860',
 '22634',
 '46.21',
 '46.10',
 '45541',
 '45111',
 '27524',
 '44800',
 '44626',
 '27605',
 '44604',
 '26040',
 '25230',
 '47605',
 '25040',
 '50800',
 '50728',
 '50542',
 '23.73',
 '50081',
 '50080',
 '23130',
 '23180',
 '23405',
 '23485',
 '23600',
 '23605',
 '23615',
 '24147',
 '49325',
 '49322',
 '25.91',
 '49255',
 '48.76',
 'V2790',
 '41120']

In [ ]:
len(top_surgery_codes)

In [ ]:
### Isolate surgery data
surgery_pat = {}
### Calculated in ../Procedures/Identify srugery prevalance
#top_surgery_codes = ['38724', '88309', '88307', '88305', '88304', '41150', '42894', '0FT44ZZ', '14301']
surgery_value = []
for val in tqdm(outcome_corrected_PCDM_PROCEDURES['PX']):
    if str(val) in top_surgery_codes:
        surgery_value.append(1)
    else:
        surgery_value.append(0)
outcome_corrected_PCDM_PROCEDURES['SURGERY'] = surgery_value

### Isolate radiation data
rad_pat = {}
top_radiation_codes = ['77427', '77336', '77263', '77301', '77338', '77300', '77386', '77470', '77290', '77280', '77412']
radiation_value = []
for val in tqdm(outcome_corrected_PCDM_PROCEDURES['PX']):
    if str(val) in top_radiation_codes:
        radiation_value.append(1)
    else:
        radiation_value.append(0)

outcome_corrected_PCDM_PROCEDURES['RADIATION'] = radiation_value

In [ ]:
surgery_radiation_data = outcome_corrected_PCDM_PROCEDURES[['PATID', 'PX_DATE','SURGERY', 'RADIATION']]

In [ ]:
surgery_radiation_data

In [ ]:
len(surgery_radiation_data[surgery_radiation_data['SURGERY']==1]['PATID'].unique())

#### Identify other procedures

In [ ]:
### other procedures
surgery_rad_codes = top_radiation_codes + top_surgery_codes
other_procedures = outcome_corrected_PCDM_PROCEDURES[~outcome_corrected_PCDM_PROCEDURES['PX'].isin(surgery_rad_codes)]
proc_data = other_procedures[['PATID', 'PX', 'PX_DATE']]
proc_data

### PCDM_MEDADMIN/PCDM_PRESCRIBING

per patient data

- chemotherapy treatment
- immunotherapy treatment

to extract medication codes:

- medication codes
- date of medication delivery

In [ ]:
### check if value is a number
def isNumber(s):
    punc = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
    for ele in s:
        if ele in punc:
            
            s = s.replace(ele, "")
            return True
    try:
        float(s)
        return True
    except:
        return False


### Run this first once to ensure that medadmin has necessary colunn
### Adds medadmin name truncated to medadmin file
# initial words list to split medications
### Takes about:  
def medicationNameListMaker(medadmin, col_name):
    words = []
    [words.append(word.split()) for word in medadmin[col_name]]
    #print(words)
    #create medication list of just the medication
    labels = [''] * len(words)
    i = 0
    print(medadmin[col_name][i])
    for word in words:
        #print(word)
        for part in word:
            #print(part)
            if (isNumber(part)):
                #print(labels[i])
                #print("end")
                break
            labels[i] += part + ' '
        print(labels[i])
        i+=1

    #Remove last space
    medications =  [label[:-1] for label in labels]

    medadmin['Medication_Name'] = medications
    return medications


### check if value is a number
def isNumber(s):
    punc = '''!()-[]{};:'"\,<>./?@#$%^&*_~'''
    for ele in s:
        if ele in punc:
            
            s = s.replace(ele, "")
            return True
    try:
        float(s)
        return True
    except:
        return False

def is_none_or_whitespace(s):
    return s is None or s == "" or str.lower(s) =='nan'

### Run this first once to ensure that medadmin has necessary colunn
### Adds medadmin name truncated to medadmin file
# initial words list to split medications
### Takes about:  
def medicationNameExtractor(raw_med_name):
    # words = []
    # [words.append(word.split()) for word in medadmin[col_name]]
    #print(words)
    #create medication list of just the medication
    try:
        str(raw_med_name)
    except:
        return ''


    if type(raw_med_name) == float:
        return ''

    if is_none_or_whitespace(raw_med_name):
        return ''
    word = ''
    for part in raw_med_name:
        if (isNumber(part)):
            #print(labels[i])
            #print("end")
            break
        word += part
    #Remove last space
    output =  word[:-1] 
    return str.lower(output)

In [ ]:
medicationRootName = []
for i,val in tqdm(enumerate(PCDM_MED_ADMIN['MEDADMIN_CODE'])):
    code_type = PCDM_MED_ADMIN.loc[i, 'MEDADMIN_TYPE']
    #print(i)
    if code_type == 'RX':
        translation = rxCodeTranslateDict[val]
        if translation == '':
            rawMed_Name = PCDM_MED_ADMIN.loc[i, 'RAW_MEDADMIN_MED_NAME']
            medicationRootName.append(medicationNameExtractor(rawMed_Name))
        else:
            medicationRootName.append(translation)
    elif code_type == 'ND':
        translation = ndcCodeTranslateDict[val]
        if translation == '':
            rawMed_Name = PCDM_MED_ADMIN.loc[i, 'RAW_MEDADMIN_MED_NAME']
            medicationRootName.append(medicationNameExtractor(rawMed_Name))
        else:
            medicationRootName.append(translation)
    else:
        rawMed_Name = PCDM_MED_ADMIN.loc[i, 'RAW_MEDADMIN_MED_NAME']
        medicationRootName.append(medicationNameExtractor(rawMed_Name))

PCDM_MED_ADMIN['TRANSLATED_MED_CODES'] = medicationRootName

In [ ]:
PCDM_PRESCRIBING = pd.read_csv('/path/to/IRB_DATA_DRIVE/PCDM_2023_03_09/IRB_PCDM_PRESCRIBING.csv', on_bad_lines='skip' ,encoding = 'latin-1', engine='python')
print("14/17 Read in PCDM_PRESCRIBING")

In [ ]:
# Define the pattern for erroneous characters
pattern = r'[@?]'
# Find rows where any column contains the pattern
PCDM_PRESCRIBING = PCDM_PRESCRIBING[~PCDM_PRESCRIBING.apply(lambda x: x.astype(str).str.contains(pattern, regex=True)).any(axis=1)]


# Convert column 'RXNORM_CUI' to numeric, forcing non-integer values to NaN
PCDM_PRESCRIBING['RXNORM_CUI'] = pd.to_numeric(PCDM_PRESCRIBING['RXNORM_CUI'], errors='coerce')

# Remove rows with NaN values in column 'RXNORM_CUI'
PCDM_PRESCRIBING = PCDM_PRESCRIBING.dropna(subset=['RXNORM_CUI']).reset_index(drop=True)

# # Convert column 'A' to numeric, forcing non-integer values to NaN
# PCDM_PRESCRIBING['RXNORM_CUI_TEST'] = pd.to_numeric(PCDM_PRESCRIBING['RXNORM_CUI'], errors='coerce')

# # Identify rows with NaN values in column 'A_numeric'
# non_integer_rows = PCDM_PRESCRIBING[PCDM_PRESCRIBING['RXNORM_CUI_TEST'].isnull()]

In [ ]:
### PREP COLUMNS
rxCodeTranslateDict['0.0'] = ''
PCDM_PRESCRIBING = PCDM_PRESCRIBING[PCDM_PRESCRIBING['RXNORM_CUI']!='N']
PCDM_PRESCRIBING['RXNORM_CUI'] = PCDM_PRESCRIBING['RXNORM_CUI'].fillna(0)
PCDM_PRESCRIBING['RXNORM_CUI']=PCDM_PRESCRIBING['RXNORM_CUI'].apply(lambda x: int(x))

### TRANSLATE MED CODES IN PRESCRIBING
prescribingMedicationRootName = []
for i,val in tqdm(enumerate(PCDM_PRESCRIBING['RXNORM_CUI'])):
    #print(float(val))
    val = str(float(val))
    translation = rxCodeTranslateDict[val]
    if translation == '':
        rawMed_Name = PCDM_PRESCRIBING.loc[i, 'RAW_RX_MED_NAME']
        #print(rawMed_Name)
        prescribingMedicationRootName.append(medicationNameExtractor(rawMed_Name))
    else:
        prescribingMedicationRootName.append(translation)

PCDM_PRESCRIBING['TRANSLATED_MED_CODES'] = prescribingMedicationRootName

#### Limit to before outcome date

In [ ]:
### outcome date correct pcdm_med_admin
print(len(PCDM_MED_ADMIN))
PCDM_MED_ADMIN['MEDADMIN_START_DATE'] = pd.to_datetime(PCDM_MED_ADMIN['MEDADMIN_START_DATE'], errors = 'coerce')
PCDM_MED_ADMIN = PCDM_MED_ADMIN.dropna(subset = ['MEDADMIN_START_DATE'])
print(len(PCDM_MED_ADMIN))
metPatients = list(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID']))
data_rows = []
parse_pat = [p for p in PCDM_MED_ADMIN['PATID'].unique() if p not in drop_pat]
for pat in tqdm(parse_pat):
    temp = PCDM_MED_ADMIN[PCDM_MED_ADMIN['PATID']==pat]
    # Get HNC diagnosis date
    hnc_diag_date = HNC_DIAG_PAT.get(pat)
    if pd.notna(hnc_diag_date):
        temp = temp[temp['MEDADMIN_START_DATE'] >= pd.to_datetime(hnc_diag_date)]
    
    ## get metastasis outcome date
    if pat in metPatients:
        outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
        append_rows = temp[temp['MEDADMIN_START_DATE']<= outcome_date]
        data_rows.append(append_rows)
    else:
        data_rows.append(temp)
    
outcome_corrected_PCDM_MED_ADMIN = pd.concat(data_rows, ignore_index=True)


In [ ]:
### OTUCOME DATE CORRECT PCDM_PRESCRIBING
print(len(PCDM_PRESCRIBING))
PCDM_PRESCRIBING['RX_START_DATE'] = pd.to_datetime(PCDM_PRESCRIBING['RX_START_DATE'], errors = 'coerce')
PCDM_PRESCRIBING = PCDM_PRESCRIBING.dropna(subset = ['RX_START_DATE'])
print(len(PCDM_PRESCRIBING))
metPatients = list(set(metastasis_data[metastasis_data['METASTASIS']==1]['PATID']))
data_rows = []
parse_pat = [p for p in PCDM_PRESCRIBING['PATID'].unique() if p not in drop_pat]
for pat in tqdm(parse_pat):
    temp = PCDM_PRESCRIBING[PCDM_PRESCRIBING['PATID']==pat]
    # Get HNC diagnosis date
    hnc_diag_date = HNC_DIAG_PAT.get(pat)
    if pd.notna(hnc_diag_date):
        temp = temp[temp['RX_START_DATE'] >= pd.to_datetime(hnc_diag_date)]

    if pat in metPatients:
        outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
        append_rows = temp[temp['RX_START_DATE']<= outcome_date]
        data_rows.append(append_rows)
    else:
        data_rows.append(temp)
    
outcome_corrected_PCDM_PRESCRIBING = pd.concat(data_rows, ignore_index=True)


#### Chemo/immuno therapy

In [ ]:
### PCDM_MED_ADMIN chemotherapy and immunotherapy
chemo_value = []
chemotherapy_medications = ['cytarabine','cisplatin', 'carboplatin', 'paclitaxel', 'docetaxel', '5-fluorouracil', '5-fu', 'fluorouracil','capecitabine', 'gemcitabine', 'doxorubicin', 'adriamycin', 'epirubicin', 'cyclophosphamide', 'cyclophosphamide', 'ifosfamide', 'etoposide', 'irinotecan', 'topotecan', 'vinorelbine', 'vinblastine', 'vincristine', 'bleomycin', 'mitomycin', 'mitoxantrone', 'oxaliplatin', 'pemetrexed', 'docetaxel', 'paclit', 'hydroxyurea', 'taxotere']
immuno_value = []
immuno_medications = ['pembrolizumab', 'toripalimab', 'nivolumab', 'avastin', 'bevacizumab', 'erbitux', 'cetuximab', 'keytruda', 'opdivo', 'yervoy', 'ipilimumab', 'revlimid', 'lenalidomide', 'thalomid', 'thalidomide']

for val in tqdm(outcome_corrected_PCDM_MED_ADMIN['TRANSLATED_MED_CODES']):
    if val in chemotherapy_medications:
        chemo_value.append(1)
    else:
        chemo_value.append(0)
    
    if val in immuno_medications:
        immuno_value.append(1)
    else:
        immuno_value.append(0)
outcome_corrected_PCDM_MED_ADMIN['CHEMOTHERAPY'] = chemo_value
outcome_corrected_PCDM_MED_ADMIN['IMMUNOTHERAPY'] = immuno_value

PCDM_MED_Admin_chemo_immuno = outcome_corrected_PCDM_MED_ADMIN[['PATID', 'MEDADMIN_START_DATE', 'TRANSLATED_MED_CODES', 'CHEMOTHERAPY', 'IMMUNOTHERAPY']]

#### PCDM_PRESCRIBING chmotherapy and immunotherapy
chemo_value = []
immuno_value = []
for val in tqdm(outcome_corrected_PCDM_PRESCRIBING['TRANSLATED_MED_CODES']):
    if val in chemotherapy_medications:
        chemo_value.append(1)
    else:
        chemo_value.append(0)
    
    if val in immuno_medications:
        immuno_value.append(1)
    else:
        immuno_value.append(0)
    
outcome_corrected_PCDM_PRESCRIBING['CHEMOTHERAPY'] = chemo_value
outcome_corrected_PCDM_PRESCRIBING['IMMUNOTHERAPY'] = immuno_value

PCDM_Prescribing_chemo_immuno = outcome_corrected_PCDM_PRESCRIBING[['PATID', 'RX_START_DATE', 'TRANSLATED_MED_CODES', 'CHEMOTHERAPY', 'IMMUNOTHERAPY']]


In [ ]:
### combine the immuno_Chemo data
PCDM_chemo_immuno = pd.concat([PCDM_MED_Admin_chemo_immuno, PCDM_Prescribing_chemo_immuno], ignore_index=True)
PCDM_chemo_immuno['DATE'] = PCDM_chemo_immuno['MEDADMIN_START_DATE'].combine_first(PCDM_chemo_immuno['RX_START_DATE'])
PCDM_chemo_immuno = PCDM_chemo_immuno.drop(columns = ['MEDADMIN_START_DATE', 'RX_START_DATE'])
PCDM_chemo_immuno = PCDM_chemo_immuno.groupby(['PATID','DATE', 'TRANSLATED_MED_CODES']).max().reset_index()
PCDM_chemo_immuno

In [ ]:
print(len(PCDM_chemo_immuno[PCDM_chemo_immuno['CHEMOTHERAPY']==1]['PATID'].unique()))
print(len(PCDM_chemo_immuno[PCDM_chemo_immuno['IMMUNOTHERAPY']==1]['PATID'].unique()))

#### Final data of other meds

In [ ]:
outcome_corrected_PCDM_MED_ADMIN

In [ ]:
outcome_corrected_PCDM_MED_ADMIN

In [ ]:
immuno_chemo = chemotherapy_medications+immuno_medications
prescrib_dat = outcome_corrected_PCDM_PRESCRIBING[~outcome_corrected_PCDM_PRESCRIBING['TRANSLATED_MED_CODES'].isin(immuno_chemo)]
med_admin_dat = outcome_corrected_PCDM_MED_ADMIN[~outcome_corrected_PCDM_MED_ADMIN['TRANSLATED_MED_CODES'].isin(immuno_chemo)]

prescrib_dat = prescrib_dat[['PATID', 'TRANSLATED_MED_CODES', 'RX_START_DATE']]
med_admin_dat = med_admin_dat[['PATID','MEDADMIN_START_DATE' ,'TRANSLATED_MED_CODES']]

In [ ]:
prescrib_dat

In [ ]:
med_admin_dat

In [ ]:
PCDM_med_data = pd.concat([prescrib_dat, med_admin_dat], ignore_index=True)
PCDM_med_data['DATE'] = PCDM_med_data['RX_START_DATE'].combine_first(PCDM_med_data['MEDADMIN_START_DATE'])
PCDM_med_data = PCDM_med_data.drop(columns = ['RX_START_DATE', 'MEDADMIN_START_DATE'])
PCDM_med_data = PCDM_med_data.groupby(['PATID','DATE', 'TRANSLATED_MED_CODES']).max().reset_index()
PCDM_med_data

### Merge it all

#### Week level data

##### Diagnosis data

In [ ]:
comorbididty_data

##### Procedures

In [ ]:
proc_data.head()

##### Medications

In [ ]:
PCDM_med_data.head()

##### Merge it together

In [ ]:
commonPats = list(set(PCDM_DIAGNOSIS['PATID'])
                   .intersection(PCDM_MED_ADMIN['PATID'])
                   .intersection(PCDM_PROCEDURES['PATID'])
                   .intersection(PCDM_PRESCRIBING['PATID'])
                )
commonPats = [p for p in commonPats if p not in drop_pat]    

In [ ]:
comorbididty_data = comorbididty_data[comorbididty_data['PATID'].isin(commonPats)]
proc_data = proc_data[proc_data['PATID'].isin(commonPats)]
PCDM_med_data = PCDM_med_data[PCDM_med_data['PATID'].isin(commonPats)]
# med_admin_dat = med_admin_dat[med_admin_dat['PATID'].isin(commonPats)]

In [ ]:
comorbididty_data['DX_DATE'] = pd.to_datetime(comorbididty_data['DX_DATE'], errors = 'coerce')
comorbididty_data = comorbididty_data.dropna(subset = ['DX_DATE'])

proc_data['PX_DATE'] = pd.to_datetime(proc_data['PX_DATE'], errors = 'coerce')
proc_data = proc_data.dropna(subset = ['PX_DATE'])

PCDM_med_data['DATE'] = pd.to_datetime(PCDM_med_data['DATE'], errors = 'coerce')
PCDM_med_data = PCDM_med_data.dropna(subset = ['DATE'])

# prescrib_dat['RX_START_DATE'] = pd.to_datetime(prescrib_dat['RX_START_DATE'], errors= 'coerce')
# prescrib_dat = prescrib_dat.dropna(subset = ['RX_START_DATE'])

In [ ]:
dataFiles = {
    'DIAGNOSIS': comorbididty_data,
    'PROCEDURES': proc_data,
    'MEDICATION': PCDM_med_data
}

In [ ]:
### Get metastasis status
onlyMetData = PCDM_DIAGNOSIS[PCDM_DIAGNOSIS['DX'].isin(metastasisICD)]
### Create list of patients with metastasis ICD codes
metPatients=list(set(onlyMetData['PATID']))

In [ ]:
dataSet = pd.DataFrame(columns = ['PATID', 'DATE_TIME', 'VALUE', 'CATEGORY', 'LABEL'])
data_rows = []
for pat in tqdm(commonPats):
    for key, file in dataFiles.items():
        if key == 'DIAGNOSIS':
            temp = file[file['PATID'] == pat]
            temp['DX_DATE'] = pd.to_datetime(temp['DX_DATE'])
            # Filter to after HNC diagnosis
            hnc_diag_date = HNC_DIAG_PAT.get(pat)
            if pd.notna(hnc_diag_date):
                temp = temp[temp['DX_DATE'] >= pd.to_datetime(hnc_diag_date)]
            
            if pat in metPatients:
                outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
                temp = temp[temp['DX_DATE'] <= outcome_date]

            diagCodeLabels = []
            for val in temp['DX']:
                diagCodeLabels.append(diagTranslateDict[val])

            appendData = {
                'PATID': temp['PATID'],
                'DATE_TIME': temp['DX_DATE'],
                'VALUE': temp['DX'],
                'CATEGORY': [key] * len(temp),
                'LABEL': diagCodeLabels
            }
            new_rows = pd.DataFrame(appendData)
            data_rows.append(new_rows)
        elif key == 'PROCEDURES':
            temp = file[file['PATID'] == pat]
            temp['PX_DATE'] = pd.to_datetime(temp['PX_DATE'])
            
            # Filter to after HNC diagnosis
            hnc_diag_date = HNC_DIAG_PAT.get(pat)
            if pd.notna(hnc_diag_date):
                temp = temp[temp['PX_DATE'] >= pd.to_datetime(hnc_diag_date)]
                        
            if pat in metPatients:
                outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
                temp = temp[temp['PX_DATE'] <= outcome_date]
            
            procedCodesLabels = []
            for val in temp['PX']:
                procedCodesLabels.append(procedTranlateDict[val])

            appendData = {
                'PATID': temp['PATID'],
                'DATE_TIME': temp['PX_DATE'],
                'VALUE': temp['PX'],
                'CATEGORY': [key] * len(temp),
                'LABEL': procedCodesLabels
            }

            new_rows = pd.DataFrame(appendData)
            data_rows.append(new_rows)
        
        elif key == 'MEDICATION':
            temp = file[file['PATID'] == pat]
            temp['DATE'] = pd.to_datetime(temp['DATE'])
            
            # Filter to after HNC diagnosis
            hnc_diag_date = HNC_DIAG_PAT.get(pat)
            if pd.notna(hnc_diag_date):
                temp = temp[temp['DATE'] >= pd.to_datetime(hnc_diag_date)]

            if pat in metPatients:
                outcome_date = metastasis_data[metastasis_data['PATID']==pat]['OUTCOME_DATE'].min()
                temp = temp[temp['DATE'] <= outcome_date]
                
            appendData = {
                'PATID': temp['PATID'],
                'DATE_TIME': temp['DATE'],
                'VALUE': temp['TRANSLATED_MED_CODES'],
                'CATEGORY': [key] * len(temp),
                'LABEL': temp['TRANSLATED_MED_CODES']
            }

            new_rows = pd.DataFrame(appendData)
            data_rows.append(new_rows)

input_data = pd.concat(data_rows, ignore_index=True) 

In [ ]:
input_data

In [ ]:
input_data.to_csv('input_data/raw_input_data.csv')

### one hot encoding

### bin per pat

In [ ]:
### Chemothearpy immunotherapy data
# PCDM_chemo_immuno = PCDM_chemo_immuno.drop(columns = ['TRANSLATED_MED_CODES'])
# PCDM_chemo_immuno = PCDM_chemo_immuno.rename(columns={'DATE': 'DATE_TIME'})
# PCDM_chemo_immuno = PCDM_chemo_immuno.set_index(['PATID', 'DATE_TIME'])
# input_data = input_data.set_index(['PATID', 'DATE_TIME'])
# input_data = input_data.join(PCDM_chemo_immuno, how='outer')
# input_data = input_data.reset_index()

# ### Surgery radiation data
# surgery_radiation_data = surgery_radiation_data.rename(columns={'PX_DATE': 'DATE_TIME'})
# surgery_radiation_data = surgery_radiation_data.set_index(['PATID', 'DATE_TIME'])
# input_data = input_data.set_index(['PATID', 'DATE_TIME'])
# input_data = input_data.join(surgery_radiation_data, how='outer')
# input_data = input_data.reset_index()

In [ ]:
# one_hot_encoded_input = pd.get_dummies(input_data, columns = ['VALUE'], prefix = '', sparse = True)

In [ ]:
# one_hot_encoded_input = one_hot_encoded_input.drop(columns=['CATEGORY', 'LABEL'])

In [ ]:
# one_hot_encoded_input

In [ ]:
# one_hot_encone_hot_encoded_input_working = one_hot_encoded_input.drop(columns = ['DATE_TIME'])

In [ ]:
# one_hot_encoded_merged = one_hot_encone_hot_encoded_input_working.groupby('PATID').max().reset_index()

In [ ]:
# one_hot_encoded_merged

In [ ]:
# PCDM_chemo_immuno.head()

In [ ]:
# surgery_radiation_data.head()

In [ ]:
# PCDM_chemo_immuno_working = PCDM_chemo_immuno.drop(columns = ['TRANSLATED_MED_CODES', 'DATE'])
# chemo_immuno_grouped = PCDM_chemo_immuno_working.groupby(['PATID']).max().reset_index()
# surgery_rad_working = surgery_radiation_data.drop(columns = ['PX_DATE'])
# surgery_radiation_grouped = surgery_rad_working.groupby(['PATID']).max().reset_index()

In [ ]:
# ### Chemothearpy immunotherapy data
# chemo_immuno_grouped = chemo_immuno_grouped.set_index(['PATID'])
# one_hot_encoded_merged = one_hot_encoded_merged.set_index(['PATID'])
# one_hot_encoded_merged = one_hot_encoded_merged.join(chemo_immuno_grouped, how='outer')
# one_hot_encoded_merged = one_hot_encoded_merged.reset_index()

# ### Surgery radiation data
# surgery_radiation_grouped = surgery_radiation_grouped.set_index(['PATID'])
# one_hot_encoded_merged = one_hot_encoded_merged.set_index(['PATID'])
# one_hot_encoded_merged = one_hot_encoded_merged.join(surgery_radiation_grouped, how='outer')
# one_hot_encoded_merged = one_hot_encoded_merged.reset_index()

In [ ]:
# one_hot_encoded_merged.to_csv('input_data/one_hot_encoded_bin_per_pat.csv')

In [ ]:
# one_hot_encoded_merged.head(15)

### sum per pat

In [ ]:
PCDM_med_data.head()

In [ ]:
comorbididty_data.head()

In [ ]:
proc_data.head()

In [ ]:
### One hot encode them
comorbidity_encoded = pd.get_dummies(comorbididty_data, columns = ['DX'], prefix = '', sparse = True)
PCDM_med_encoded = pd.get_dummies(PCDM_med_data, columns = ['TRANSLATED_MED_CODES'], prefix = '', sparse = True)
proc_data_encoded = pd.get_dummies(proc_data, columns = ['PX'], prefix = '', sparse = True)

In [ ]:
comorbidity_encoded_grouped = comorbidity_encoded.drop(columns = ['DX_DATE'])
parsing_columns = [col for col in comorbidity_encoded_grouped.columns if col != 'PATID']
for col in tqdm(parsing_columns):
    comorbidity_encoded_grouped[col] = comorbidity_encoded_grouped[col].astype(int)
comorbidity_encoded_grouped = comorbidity_encoded_grouped.groupby(['PATID']).sum().reset_index()

In [ ]:
PCDM_med_encoded_grouped = PCDM_med_encoded.drop(columns = ['DATE'])
parsing_columns = [col for col in PCDM_med_encoded_grouped.columns if col != 'PATID']
for col in tqdm(parsing_columns):
    PCDM_med_encoded_grouped[col] = PCDM_med_encoded_grouped[col].astype(int)
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.groupby(['PATID']).sum().reset_index()

In [ ]:
proc_data_encoded_grouped = proc_data_encoded.drop(columns = ['PX_DATE'])
parsing_columns = [col for col in proc_data_encoded_grouped.columns if col != 'PATID']
for col in tqdm(parsing_columns):
    proc_data_encoded_grouped[col] = proc_data_encoded_grouped[col].astype(int)
proc_data_encoded_grouped = proc_data_encoded_grouped.groupby(['PATID']).sum().reset_index()

In [ ]:
### comorbidity data data
comorbidity_encoded_grouped = comorbidity_encoded_grouped.set_index(['PATID'])

### medication data
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.set_index(['PATID'])

### procedure data
proc_data_encoded_grouped = proc_data_encoded_grouped.set_index(['PATID'])

### join the data
working_data = comorbidity_encoded_grouped.join(PCDM_med_encoded_grouped, how='outer')
working_data = working_data.join(proc_data_encoded_grouped, how='outer')
working_data = working_data.reset_index()

comorbidity_encoded_grouped = comorbidity_encoded_grouped.reset_index()
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.reset_index()
proc_data_encoded_grouped = proc_data_encoded_grouped.reset_index()
working_data = working_data.replace(np.nan, 0)

In [ ]:
working_data.head()

In [ ]:
working_data.to_csv('input_data/one_hot_encoded_sum_per_pat_no_confounder.csv')

In [ ]:
PCDM_chemo_immuno_working = PCDM_chemo_immuno.drop(columns = ['TRANSLATED_MED_CODES', 'DATE'])
chemo_immuno_grouped = PCDM_chemo_immuno_working.groupby(['PATID']).sum().reset_index()
surgery_rad_working = surgery_radiation_data.drop(columns = ['PX_DATE'])
surgery_radiation_grouped = surgery_rad_working.groupby(['PATID']).sum().reset_index()

In [ ]:
one_hot_encoded_merged = working_data

In [ ]:
one_hot_encoded_merged.head()

In [ ]:
### Chemothearpy immunotherapy data
chemo_immuno_grouped = chemo_immuno_grouped.set_index(['PATID'])
one_hot_encoded_merged = one_hot_encoded_merged.set_index(['PATID'])
one_hot_encoded_merged = one_hot_encoded_merged.join(chemo_immuno_grouped, how='outer')
one_hot_encoded_merged = one_hot_encoded_merged.reset_index()

### Surgery radiation data
surgery_radiation_grouped = surgery_radiation_grouped.set_index(['PATID'])
one_hot_encoded_merged = one_hot_encoded_merged.set_index(['PATID'])
one_hot_encoded_merged = one_hot_encoded_merged.join(surgery_radiation_grouped, how='outer')
one_hot_encoded_merged = one_hot_encoded_merged.reset_index()

In [ ]:
one_hot_encoded_merged

In [ ]:
# Check if any metastasis date is before HNC diagnosis date
patient_data= pd.read_csv('input_data/patient level data.csv')
sanity_check = patient_data[patient_data['OUTCOME_DATE'] < patient_data['HNC_DIAG_DATE']]
print(f"Number of patients with metastasis before diagnosis: {len(sanity_check)}")
sanity_check[['PATID', 'HNC_DIAG_DATE', 'OUTCOME_DATE', 'METASTASIS']]
outcome_before_diag = sanity_check[sanity_check['OUTCOME_DATE'] < sanity_check['HNC_DIAG_DATE']]
outcome_before_diag_pat = list(set(outcome_before_diag['PATID']))
print(f"Number of patients with metastasis outcome before diagnosis: {len(outcome_before_diag)}")
print(f"Patients with metastasis outcome before diagnosis: {outcome_before_diag_pat}")

In [ ]:
### confirming no patients have metastasis outcome before diagnosis in the one hot encoded data
one_hot_encoded_merged[one_hot_encoded_merged['PATID'].isin(outcome_before_diag_pat)]

In [ ]:
one_hot_encoded_merged.to_csv('input_data/one_hot_encoded_sum_per_pat.csv')

In [ ]:
stop

##### no proc

In [ ]:
PCDM_med_data.head()

In [ ]:
comorbididty_data.head()

In [ ]:
### One hot encode them
comorbidity_encoded = pd.get_dummies(comorbididty_data, columns = ['DX'], prefix = '', sparse = True)
PCDM_med_encoded = pd.get_dummies(PCDM_med_data, columns = ['TRANSLATED_MED_CODES'], prefix = '', sparse = True)

In [ ]:
comorbidity_encoded_grouped = comorbidity_encoded.drop(columns = ['DX_DATE'])
parsing_columns = [col for col in comorbidity_encoded_grouped.columns if col != 'PATID']
for col in tqdm(parsing_columns):
    comorbidity_encoded_grouped[col] = comorbidity_encoded_grouped[col].astype(int)
comorbidity_encoded_grouped = comorbidity_encoded_grouped.groupby(['PATID']).sum().reset_index()

In [ ]:
PCDM_med_encoded_grouped = PCDM_med_encoded.drop(columns = ['DATE'])
parsing_columns = [col for col in PCDM_med_encoded_grouped.columns if col != 'PATID']
for col in tqdm(parsing_columns):
    PCDM_med_encoded_grouped[col] = PCDM_med_encoded_grouped[col].astype(int)
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.groupby(['PATID']).sum().reset_index()

In [ ]:
### comorbidity data data
comorbidity_encoded_grouped = comorbidity_encoded_grouped.set_index(['PATID'])

### medication data
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.set_index(['PATID'])

### join the data
working_data = comorbidity_encoded_grouped.join(PCDM_med_encoded_grouped, how='outer')
working_data = working_data.reset_index()

comorbidity_encoded_grouped = comorbidity_encoded_grouped.reset_index()
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.reset_index()
working_data = working_data.replace(np.nan, 0)

In [ ]:
working_data.head()

In [ ]:
#### Confounders
PCDM_chemo_immuno_working = PCDM_chemo_immuno.drop(columns = ['TRANSLATED_MED_CODES', 'DATE'])
chemo_immuno_grouped = PCDM_chemo_immuno_working.groupby(['PATID']).sum().reset_index()
surgery_rad_working = surgery_radiation_data.drop(columns = ['PX_DATE'])
surgery_radiation_grouped = surgery_rad_working.groupby(['PATID']).sum().reset_index()

In [ ]:
chemo_immuno_grouped

In [ ]:
surgery_radiation_grouped

In [ ]:
one_hot_encoded_merged_no_proc = working_data

In [ ]:
one_hot_encoded_merged_no_proc.head()

In [ ]:
### Chemothearpy immunotherapy data
chemo_immuno_grouped = chemo_immuno_grouped.set_index(['PATID'])
one_hot_encoded_merged_no_proc = one_hot_encoded_merged_no_proc.set_index(['PATID'])
one_hot_encoded_merged_no_proc = one_hot_encoded_merged_no_proc.join(chemo_immuno_grouped, how='outer')
one_hot_encoded_merged_no_proc = one_hot_encoded_merged_no_proc.reset_index()

### Surgery radiation data
surgery_radiation_grouped = surgery_radiation_grouped.set_index(['PATID'])
one_hot_encoded_merged_no_proc = one_hot_encoded_merged_no_proc.set_index(['PATID'])
one_hot_encoded_merged_no_proc = one_hot_encoded_merged_no_proc.join(surgery_radiation_grouped, how='outer')
one_hot_encoded_merged_no_proc = one_hot_encoded_merged_no_proc.reset_index()

In [ ]:
one_hot_encoded_merged_no_proc.to_csv('input_data/one_hot_encoded_sum_per_pat_no_proc.csv')
one_hot_encoded_merged_no_proc.head()

#### No proc/ No diag

In [ ]:
PCDM_med_data

In [ ]:
### One hot encode them
PCDM_med_encoded = pd.get_dummies(PCDM_med_data, columns = ['TRANSLATED_MED_CODES'], prefix = '', sparse = True)

In [ ]:
PCDM_med_encoded_grouped = PCDM_med_encoded.drop(columns = ['DATE'])
parsing_columns = [col for col in PCDM_med_encoded_grouped.columns if col != 'PATID']
for col in tqdm(parsing_columns):
    PCDM_med_encoded_grouped[col] = PCDM_med_encoded_grouped[col].astype(int)
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.groupby(['PATID']).sum().reset_index()

In [ ]:
### medication data
PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.set_index(['PATID'])

### join the data
working_data = PCDM_med_encoded_grouped

PCDM_med_encoded_grouped = PCDM_med_encoded_grouped.reset_index()
working_data = working_data.replace(np.nan, 0)

In [ ]:
working_data.head()

In [ ]:
#### Confounders
PCDM_chemo_immuno_working = PCDM_chemo_immuno.drop(columns = ['TRANSLATED_MED_CODES', 'DATE'])
chemo_immuno_grouped = PCDM_chemo_immuno_working.groupby(['PATID']).sum().reset_index()
surgery_rad_working = surgery_radiation_data.drop(columns = ['PX_DATE'])
surgery_radiation_grouped = surgery_rad_working.groupby(['PATID']).sum().reset_index()

In [ ]:
one_hot_encoded_merged_no_proc_no_diag = working_data

In [ ]:
one_hot_encoded_merged_no_proc_no_diag.reset_index()

In [ ]:
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.reset_index()
### Chemothearpy immunotherapy data
chemo_immuno_grouped = chemo_immuno_grouped.reset_index()
chemo_immuno_grouped = chemo_immuno_grouped.set_index(['PATID'])
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.set_index(['PATID'])
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.join(chemo_immuno_grouped, how='outer')
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.reset_index()

### Surgery radiation data
surgery_radiation_grouped.reset_index()
surgery_radiation_grouped = surgery_radiation_grouped.set_index(['PATID'])
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.set_index(['PATID'])
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.join(surgery_radiation_grouped, how='outer')
one_hot_encoded_merged_no_proc_no_diag = one_hot_encoded_merged_no_proc_no_diag.reset_index()

In [ ]:
one_hot_encoded_merged_no_proc_no_diag.to_csv('input_data/one_hot_encoded_sum_per_pat_no_proc_no_diag.csv')
one_hot_encoded_merged_no_proc_no_diag.head()

In [ ]:
one_hot_encoded_merged_no_proc_no_diag